# 1 Module, Globals und Daten

## 1.1 Module

In [ ]:
##### IMPORTS #####

import numpy as np
import pandas as pd

from xgboost import XGBClassifier

import sklearn.metrics as skm
from sklearn.model_selection import train_test_split, learning_curve

import os

import matplotlib.pyplot as pyplot

## 1.2 Globals

In [ ]:
##### Globals #####
dataPath = 'datasets/gyro/'                         # Set location of dataset


## 1.3 Daten

In [ ]:
##### Data Import and Splitting #####
data = pd.read_csv(dataPath + 'gyro_mobile.csv')    # Dataset is imbalanced with only ~1.7% of all labels being 0's
data = data.drop(columns='timestamp')
xdata = data.iloc[:,:6]
ydata = data.iloc[:,6:]

xtrain, xvaltest, ytrain, yvaltest = train_test_split( 
    xdata,
    ydata,
    random_state=0,
    train_size=0.33,
    stratify=ydata                                  # Preserve label imbalance across train- and test datasets
)

xval, xtest, yval, ytest = train_test_split( 
    xvaltest,
    yvaltest,
    random_state=0,
    train_size=0.5,
    stratify=yvaltest                                  # Preserve label imbalance across train- and test datasets
)

ev_val = [(xval,yval)]
ev_all = [(xtrain,ytrain),(xval,yval),(xtest,ytest)]

## 1.4 Funktionen

In [ ]:
def plotReport(model):
    global xtest, ytest
    yhat = model.predict(xtest)
    print(f'\nAccuracy Score: \t\t{skm.accuracy_score(ytest, yhat)}')          # 0.9828310161425772 stratified | 0.981664644956611  naive  | 0.9534384622562284 scale_pos_weight = len(0s)/len(1s)
    print(f'Balanced Accuracy-Score: \t{skm.balanced_accuracy_score(ytest, yhat)}') # 0.5990495790838934 stratified | 0.5770402535045962 naive  | 0.8314600386751905 scale_pos_weight = len(0s)/len(1s)
    print(f'ROC AUC Score: {skm.roc_auc_score(ytest, model.predict_proba(xtest)[:, 1])}')
    print()
    print(skm.classification_report(ytest,yhat, labels=[0,1], target_names=['0: standing','1: walking']))

def plotConfusionMatrices(model):
    skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap='Blues', normalize='true')
    pyplot.title('')
    pyplot.show()

    skm.ConfusionMatrixDisplay.from_estimator(model, xtest, ytest, cmap="Blues" ,values_format='d')
    pyplot.show()

def plotLearningCurves(model):
    
    # https://xgboosting.com/xgboost-plot-learning-curve/

    global xtest, ytest
    
    # Calculate learning curves
    train_sizes, train_scores, test_scores = learning_curve(
    estimator=model, X=xdata, y=ydata, cv=5, scoring='accuracy',
    train_sizes=np.linspace(0.75, 1.0, 10))

    # Calculate mean and standard deviation of scores
    train_mean = np.mean(train_scores, axis=1)
    train_std = np.std(train_scores, axis=1)
    test_mean = np.mean(test_scores, axis=1)
    test_std = np.std(test_scores, axis=1)

    # Plot learning curves
    pyplot.figure(figsize=(8, 6))
    pyplot.plot(train_sizes, train_mean, color='blue', marker='o', label='Training accuracy')
    pyplot.fill_between(train_sizes, train_mean + train_std, train_mean - train_std, alpha=0.15, color='blue')
    pyplot.plot(train_sizes, test_mean, color='green', marker='+', label='Validation accuracy')
    pyplot.fill_between(train_sizes, test_mean + test_std, test_mean - test_std, alpha=0.15, color='green')
    pyplot.title('Learning Curves')
    pyplot.xlabel('Training set size')
    pyplot.ylabel('Accuracy')
    pyplot.legend(loc='lower right')
    pyplot.show()

def plotFeatureImportances(model):    
    iScores = model.feature_importances_
    fNames = model.feature_names_in_

    fig,ax = pyplot.subplots()
    ax.bar(fNames, iScores)
    ax.set_ylabel("Importance Scores")
    ax.set_xlabel("Features")

    pyplot.show()

def plotGeneralizationCurves(model, custom_names = None):
    results = model.evals_result()
    names = list(model.evals_result())
    lossValue = list(results[names[0]])[0]
    
    if custom_names == None:
        for i in names:
            pyplot.plot(results[i][lossValue], label=i)
    else:
        name_index = 0
        for i in names:
            pyplot.plot(results[i][lossValue], label=custom_names[name_index])
            name_index += 1

    pyplot.ylabel(lossValue)
    pyplot.xlabel('Iterations')
    pyplot.legend()
    pyplot.grid()
    pyplot.show()

def plotROC(model):
    global xtest, ytest
    
    y_pred_proba = model.predict_proba(xtest)[:, 1]
    
    fpr, tpr, threshold = skm.roc_curve(ytest, y_pred_proba)
    roc_auc = skm.auc(fpr, tpr)

    pyplot.figure(figsize=(8, 6))
    pyplot.plot(fpr, tpr, color='blue', label=f'ROC curve (AUC = {roc_auc:.4f})')
    pyplot.plot([0, 1], [0, 1], color='red', linestyle='--', label='Random guess')
    pyplot.xlim([0.0, 1.0])
    pyplot.ylim([0.0, 1.05])
    pyplot.xlabel('False Positive Rate')
    pyplot.ylabel('True Positive Rate')
    pyplot.title('Receiver Operating Characteristic (ROC) Curve')
    pyplot.legend(loc="lower right")
    pyplot.show()

def checkOverfitting(model):
    global xtrain, ytrain, xtest, ytest
    print(f'\nComparing Accuracy Scores of Training and Testing Data...')
    print("Flat Scores:")
    train_acc = skm.accuracy_score(ytrain, model.predict(xtrain))
    test_acc = skm.accuracy_score(ytest, model.predict(xtest))
    print(f'Accuracy Score mit Trainings-Datensatz: {train_acc}')
    print(f'Accuracy Score mit Test-Datensatz: {test_acc}')
    print(f'Train-Test-Difference: {train_acc - test_acc}\n')

    print("Balanced Scores:")
    train_acc = skm.balanced_accuracy_score(ytrain, model.predict(xtrain))
    test_acc = skm.balanced_accuracy_score(ytest, model.predict(xtest))
    print(f'Accuracy Score mit Trainings-Datensatz: {train_acc}')
    print(f'Accuracy Score mit Test-Datensatz: {test_acc}')
    print(f'Train-Test-Difference: {train_acc - test_acc}')

def getBestDepth(listOfAccuracies, listOfSizes_kB):  
    data = {'Accuracy': listOfAccuracies,
            'Size': listOfSizes_kB}
    AccVsSize = pd.DataFrame(data)
    Suitable = AccVsSize[AccVsSize['Size']<1024]
    bestAccsWithinBoundaries = Suitable[Suitable['Accuracy']==max(Suitable['Accuracy'])]
    bestDepth = bestAccsWithinBoundaries[bestAccsWithinBoundaries['Size'] == min(bestAccsWithinBoundaries['Size'])]

    return bestDepth

# 2 Modell

## 2.1 Naives Modell / Raw Data

In [ ]:
clf_naive = XGBClassifier(
    objective = "binary:logistic",
    tree_method = 'exact',
    n_estimators = 100000,
    early_stopping_rounds = 100,
    )


clf_naive.fit(
    xtrain, ytrain,
    eval_set = ev_val,
    verbose = 0
)

print(f'Best Iteration based on Test-Data: {clf_naive.best_iteration}')

plotGeneralizationCurves(clf_naive,"Validation")

new_n_estimators = clf_naive.early_stopping_rounds + clf_naive.best_iteration

clf_naive.set_params(
    n_estimators = new_n_estimators,
    early_stopping_rounds = None
)
clf_naive.fit(
    xtrain, ytrain,
    eval_set = ev_all,
    verbose = 0
)

plotConfusionMatrices(clf_naive)
plotGeneralizationCurves(clf_naive,["Training","validation","Test"])
plotROC(clf_naive)
plotReport(clf_naive)
checkOverfitting(clf_naive)

## 2.2 Angepasstes Modell: Skalierte Klassen

[XGBoost for Imbalanced Classification](https://xgboosting.com/xgboost-scale_pos_weight-vs-sample_weight-for-imbalanced-classification/)

In [ ]:
classratio = len(data[data['Activity']==0]) / len(data[data['Activity']==1])

clf_scaled = XGBClassifier(
    objective = "binary:logistic",
    tree_method = 'exact',
    scale_pos_weight = classratio, # Increase weight of minority class
    n_estimators = 100000,
    early_stopping_rounds = 100,
)

clf_scaled.fit(xtrain,ytrain,
               eval_set=ev_val,
               verbose = 0)

print(f'Best Iteration based on Test-Data: {clf_scaled.best_iteration}')
plotGeneralizationCurves(clf_scaled,"Validation")

new_n_estimators = clf_scaled.early_stopping_rounds + clf_scaled.best_iteration

bestIteration = clf_scaled.best_iteration

clf_scaled.set_params(
    n_estimators = new_n_estimators,
    early_stopping_rounds = None
)
clf_scaled.fit(
    xtrain, ytrain,
    eval_set = ev_all,
    verbose = 0
)
plotConfusionMatrices(clf_scaled)
plotGeneralizationCurves(clf_scaled,["Training","validation","Test"])
checkOverfitting(clf_scaled)
plotROC(clf_scaled)
plotReport(clf_scaled)

## 2.2 Skaliertes Modell mit Best Iteration

In [ ]:
clf_scaled_bi = XGBClassifier(
    objective = "binary:logistic",
    tree_method = 'exact',
    scale_pos_weight = classratio, # Increase weight of minority class
    n_estimators = bestIteration,
    )

clf_scaled_bi.fit(
    xtrain, ytrain,
    eval_set = ev_all,
    verbose = 0
)
print(f'Amount of Estimators: \t\t{bestIteration}')
plotConfusionMatrices(clf_scaled_bi)
plotGeneralizationCurves(clf_scaled_bi,["Training","validation","Test"])
plotROC(clf_scaled_bi)
plotReport(clf_scaled_bi)
checkOverfitting(clf_scaled)

## 2.4 Modellverhalten & -größe bei variierender Baumtiefe

### 2.4.1 Modelle erzeugen und Daten sammeln

In [ ]:
bestIterList = []
accList = []
ubjSizeList = []
jsonSizeList = []

maximum = 30

for i in range(1,maximum+1):
    clf_iter = XGBClassifier(objective = "binary:logistic",
                            tree_method = 'exact',
                            scale_pos_weight = classratio, # Increase weight of minority class 
                            n_estimators = 100000,
                            early_stopping_rounds = 50,
                            max_depth = i
                            )

    clf_iter.fit(xtrain,ytrain,
                eval_set=ev_val,
                verbose = 0
                )

    bestIter_local = clf_iter.best_iteration
    acc_local = skm.accuracy_score(ytest,clf_iter.predict(xtest, iteration_range=(0,clf_iter.best_iteration)))
    
    clf_iter = XGBClassifier(objective = "binary:logistic",
                            tree_method = 'exact',
                            scale_pos_weight = classratio, # Increase weight of minority class 
                            n_estimators = bestIter_local,
                            early_stopping_rounds = None,
                            max_depth = i
                            )

    clf_iter.fit(xtrain,ytrain,
                eval_set=ev_all,
                verbose = 0
                )

    clf_iter.save_model('clf_iter.ubj')
    clf_iter.save_model('clf_iter.json')
    
    ubjSizeList.append(os.path.getsize('clf_iter.ubj'))
    jsonSizeList.append(os.path.getsize('clf_iter.json'))
    bestIterList.append(bestIter_local)
    accList.append(acc_local)

### 2.4.2 Gesammelte Daten visualisieren

In [ ]:
# Plot Accuracy vs Complexity

ubjSizeList_kB = [x/1024 for x in ubjSizeList]

fig, ax = pyplot.subplots(3,2)

fig.set_figwidth(12)
fig.set_figheight(12)

final_depth = getBestDepth(accList, ubjSizeList_kB).index[0]+1

##### OBEN LINKS #####

ax[0][0].set_title("Accuracy & n_estimators vs max_depth")

ax[0][0].axvline(x=final_depth, color="black", linestyle=":")

ax[0][0].set_xlabel("Maximum Tree Depth")
ax[0][0].set_ylabel("n_Estimators (Best Iteration)")
ax[0][0].tick_params(axis='y', labelcolor="blue")
ax[0][0].plot(range(1,maximum+1),bestIterList,color="blue")

ax10_twin = ax[0][0].twinx()

ax10_twin.set_ylabel("Accuracy Score")
ax10_twin.tick_params(axis='y', labelcolor="red")
ax10_twin.plot(range(1,maximum+1),accList,color="red")

##### OBEN RECHTS #####

ax[0][1].set_title("Zoomed: Accuracy & n_estimators vs max_depth[2:{maximum}]")

ax[0][1].axvline(x=final_depth, color="black", linestyle=":")

ax[0][1].set_xlabel("Maximum Tree Depth")
ax[0][1].set_ylabel("Best Iteration (n_estimators)")
ax[0][1].tick_params(axis='y', labelcolor="blue")
ax[0][1].plot(range(2,maximum+1),bestIterList[1:],color="blue")

ax11_twin = ax[0][1].twinx()

ax11_twin.set_ylabel("Accuracy Score")
ax11_twin.tick_params(axis='y', labelcolor="red")
ax11_twin.plot(range(2,maximum+1),accList[1:],color="red")

##### MITTE LINKS #####

ax[1][0].set_title("#Estimators & Model Size VS max_depth")

ax[1][0].axvline(x=final_depth, color="black", linestyle=":")

ax[1][0].set_xlabel("Maximum Tree Depth")
ax[1][0].set_ylabel("n_Estimators (Best Iteration)")
ax[1][0].tick_params(axis='y', labelcolor="blue")
ax[1][0].plot(range(1,maximum+1),bestIterList,color="blue")

ax00_twin = ax[1][0].twinx()

ax00_twin.set_ylabel("Model Size in kBytes")
ax00_twin.tick_params(axis='y', labelcolor="red")
ax00_twin.plot(range(1,maximum+1),ubjSizeList_kB,color="red")

##### MITTE RECHTS #####

ax[1][1].set_title("Zoomed: #Estimators & Model Size vs max_depth[2:{maximum}]")
# ax[1][1].set_title("Zoomed: #Estimators & Model Size vs max_depth[3:15]")

ax[1][1].axvline(x=final_depth, color="black", linestyle=":")

ax[1][1].set_xlabel("Maximum Tree Depth")
ax[1][1].set_ylabel("Bn_Estimators (Best Iteration)")
ax[1][1].tick_params(axis='y', labelcolor="blue")
ax[1][1].plot(range(2,maximum+1),bestIterList[1:],color="blue")
# ax[1][1].plot(range(3,maximum+1),bestIterList[2:],color="blue")

ax01_twin = ax[1][1].twinx()

ax01_twin.set_ylabel("Model Size in kBytes")
ax01_twin.tick_params(axis='y', labelcolor="red")
ax01_twin.plot(range(2,maximum+1),ubjSizeList_kB[1:],color="red")
# ax01_twin.plot(range(3,maximum+1),ubjSizeList_kB[2:],color="red")

##### UNTEN LINKS #####

ax[2][0].set_title("Accuracy & Model Size VS max_depth")

ax[2][0].set_xlabel("Maximum Tree Depth")
ax[2][0].set_ylabel("Accuracy Score")
ax[2][0].tick_params(axis='y', labelcolor="blue")
ax[2][0].plot(range(1,maximum+1),accList,color="blue")

ax20_twin = ax[2][0].twinx()

ax20_twin.set_ylabel("Model Size in kBytes")
ax20_twin.tick_params(axis='y', labelcolor="red")
ax20_twin.plot(range(1,maximum+1),ubjSizeList_kB,color="red")

ax20_twin.axhline(y=1000, color="red", linestyle=":", label='Size Limit (1024kB)')

ax[2][0].axvline(x=final_depth, color="black", linestyle=":")

##### UNTEN RECHTS #####

ax[2][1].set_title("Zoomed: Accuracy & Model Size vs max_depth[2:15]")
# ax[2][1].set_title("Zoomed: #Estimators & Model Size vs max_depth[3:15]")

ax[2][1].set_xlabel("Maximum Tree Depth")
ax[2][1].set_ylabel("Accuracy Score")
ax[2][1].tick_params(axis='y', labelcolor="blue")
ax[2][1].plot(range(6,maximum+1),accList[5:],color="blue")
# ax[2][1].plot(range(3,maximum+1),accList[2:],color="blue")

ax21_twin = ax[2][1].twinx()

ax21_twin.set_ylabel("Model Size in kBytes")
ax21_twin.tick_params(axis='y', labelcolor="red")
ax21_twin.plot(range(6,maximum+1),ubjSizeList_kB[5:],color="red")
# ax21_twin.plot(range(3,maximum+1),ubjSizeList_kB[2:],color="red")

ax21_twin.axhline(y=1000, color="red", linestyle=":", label="Size Limit (1024kB)")

ax[2][1].axvline(x=final_depth, color="black", linestyle=":")

##### PLOT THAT THING #####
ax21_twin.legend()

##### PLOT THAT THING #####

fig.tight_layout()
pyplot.show()


## 2.5 Basismodell mit Best Iteration und max_depth

In [ ]:
clf_final = XGBClassifier(objective = "binary:logistic",
                            tree_method = 'exact',
                            scale_pos_weight = classratio, # Increase weight of minority class 
                            n_estimators = bestIterList[final_depth-1],
                            max_depth = final_depth
                            )

clf_final.fit(xtrain,ytrain,
              eval_set=ev_all,
              verbose = 0
              )

print(f'Anzahl Estimators: {clf_final.n_estimators}')
print(f'Maximale Baumtiefe: {clf_final.max_depth}')

plotConfusionMatrices(clf_final)
plotGeneralizationCurves(clf_final,["Training","Validation","Test"])
plotROC(clf_final)
plotReport(clf_final)

print(f'Accuracy Score on Testdata: \t\t\t{skm.accuracy_score(ytest, clf_final.predict(xtest))}')
print(f'Accuracy Score on Trainingdata: \t\t{skm.accuracy_score(ytrain, clf_final.predict(xtrain))}')
print(f'Accuracy Difference between Training and Test: \t{skm.accuracy_score(ytrain, clf_final.predict(xtrain)) - skm.accuracy_score(ytest, clf_final.predict(xtest))}')

print(f'Balanced Accuracy Score on Testdata: \t\t\t{skm.balanced_accuracy_score(ytest, clf_final.predict(xtest))}')
print(f'Balanced Accuracy Score on Trainingdata: \t\t{skm.balanced_accuracy_score(ytrain, clf_final.predict(xtrain))}')
print(f'Balanced Accuracy Difference between Training and Test:\t{skm.balanced_accuracy_score(ytrain, clf_final.predict(xtrain)) - skm.balanced_accuracy_score(ytest, clf_final.predict(xtest))}')

## 2.6 Finales Basismodell sichern

In [ ]:
clf_final.save_model("model-exports/gyro_base.ubj")